In [6]:
# Block 1: load raw data files
import pandas as pd
base = 'C:/Users/cigde/Dropbox/Habituation_paper/October_Emily/'
inp, lab = base + 'input_arrays/pls_regression/', base + 'final_figures/'
erp  = pd.read_csv(inp + 'cluster_RS_all_ages.csv',   dtype={'UniqueID': str})
comp = pd.read_csv(inp + 'component_RS_all_ages.csv', dtype={'UniqueID': str})
spec = pd.read_csv(inp + 'spectral_RS_all_ages.csv',  dtype={'UniqueID': str})

In [7]:
# Block 2: build temporal feature table
temporal = erp.merge(comp, on='UniqueID')
cols = [c for c in temporal.columns if c.startswith('frontocentral_') or (c.endswith('_diff') and not c.startswith('N2'))]
temporal = temporal[['UniqueID'] + cols]

In [8]:
# Block 3: build power and phase feature tables
power = spec[['UniqueID'] + [c for c in spec.columns if c.startswith('power_') and c.endswith('poststim_diff')]]
phase = spec[['UniqueID'] + [c for c in spec.columns if c.startswith('itpc_') and c.endswith('poststim_diff')]]

In [9]:
# Block 4: define summary function and run temporal k-means clusters
def summary(label_file, df):
    m = pd.read_csv(lab + label_file, dtype={'UniqueID': str}).merge(df, on='UniqueID').drop(columns='UniqueID')
    print(m.groupby('cluster').agg(['mean', 'std']).T.round(3))

summary('labels_temporal_hierarchical.csv', temporal)

cluster                           0       1
frontocentral_W50_100  mean  -0.749   0.012
                       std    0.555   0.584
frontocentral_W100_150 mean  -0.791   0.139
                       std    0.550   0.635
frontocentral_W150_200 mean  -0.736   0.151
                       std    0.751   0.851
frontocentral_W200_250 mean  -1.157  -0.182
                       std    0.770   0.878
frontocentral_W250_300 mean  -1.281  -0.340
                       std    0.912   0.958
P1_amp_diff            mean  -1.247   0.128
                       std    1.055   0.993
N1_amp_diff            mean  -1.575   0.181
                       std    1.003   1.146
P2_amp_diff            mean  -1.203   0.394
                       std    1.294   1.250
P1_lat_diff            mean  -1.629   0.776
                       std    9.045   9.336
N1_lat_diff            mean   0.773  -0.104
                       std    9.380  12.256
P2_lat_diff            mean   3.417   2.567
                       std   11.

In [10]:
# Block 5: power k-means clusters
summary('labels_spectral_hierarchical_power_only.csv', power)

cluster                                  0      1
power_slow_poststim_diff       mean  0.030  0.449
                               std   0.650  0.659
power_fast_theta_poststim_diff mean -0.234  0.822
                               std   0.638  0.699
power_alpha_poststim_diff      mean -0.316  0.727
                               std   0.638  0.609
power_low_beta_poststim_diff   mean -0.178  0.187
                               std   0.488  0.670
power_high_beta_poststim_diff  mean -0.022 -0.011
                               std   0.536  0.555


In [11]:
# Block 6: phase k-means clusters
summary('labels_spectral_hierarchical_phase_only.csv', phase)

cluster                                 0      1
itpc_slow_poststim_diff       mean  0.000  0.019
                              std   0.055  0.060
itpc_fast_theta_poststim_diff mean -0.018  0.041
                              std   0.045  0.041
itpc_alpha_poststim_diff      mean -0.015  0.044
                              std   0.042  0.036
itpc_low_beta_poststim_diff   mean -0.005  0.018
                              std   0.040  0.032
itpc_high_beta_poststim_diff  mean -0.003  0.006
                              std   0.032  0.027


In [12]:
# Block 7: raw means table for all feature sets
import os
save_path = 'C:/Users/cigde/Dropbox/Habituation_paper/October_Emily/final_figures/supplementary/suppl_fig_3/'
os.makedirs(save_path, exist_ok=True)

rows = []
for f, df in [('labels_temporal_hierarchical.csv', temporal),
              ('labels_spectral_hierarchical_power_only.csv', power),
              ('labels_spectral_hierarchical_phase_only.csv', phase)]:
    m = pd.read_csv(lab + f, dtype={'UniqueID': str}).merge(df, on='UniqueID').drop(columns='UniqueID')
    g = m.groupby('cluster').agg(['mean', 'std'])
    for feat in m.columns.drop('cluster'):
        rows.append({'Feature': feat,
                     'Cluster 1': f"{g.loc[0, (feat, 'mean')]:.3f} ± {g.loc[0, (feat, 'std')]:.3f}",
                     'Cluster 2': f"{g.loc[1, (feat, 'mean')]:.3f} ± {g.loc[1, (feat, 'std')]:.3f}"})
table = pd.DataFrame(rows)
table.to_csv(save_path + 'raw_means_hierarchical_clusters.csv', index=False)
print(table.to_string(index=False))

                       Feature      Cluster 1       Cluster 2
         frontocentral_W50_100 -0.749 ± 0.555   0.012 ± 0.584
        frontocentral_W100_150 -0.791 ± 0.550   0.139 ± 0.635
        frontocentral_W150_200 -0.736 ± 0.751   0.151 ± 0.851
        frontocentral_W200_250 -1.157 ± 0.770  -0.182 ± 0.878
        frontocentral_W250_300 -1.281 ± 0.912  -0.340 ± 0.958
                   P1_amp_diff -1.247 ± 1.055   0.128 ± 0.993
                   N1_amp_diff -1.575 ± 1.003   0.181 ± 1.146
                   P2_amp_diff -1.203 ± 1.294   0.394 ± 1.250
                   P1_lat_diff -1.629 ± 9.045   0.776 ± 9.336
                   N1_lat_diff  0.773 ± 9.380 -0.104 ± 12.256
                   P2_lat_diff 3.417 ± 11.821  2.567 ± 14.728
      power_slow_poststim_diff  0.030 ± 0.650   0.449 ± 0.659
power_fast_theta_poststim_diff -0.234 ± 0.638   0.822 ± 0.699
     power_alpha_poststim_diff -0.316 ± 0.638   0.727 ± 0.609
  power_low_beta_poststim_diff -0.178 ± 0.488   0.187 ± 0.670
 power_h